In [5]:
import os

# Filter pptx_files for those with a title containing any of the specified keywords
# Automatically select the most recent folder in pptx_dir's parent with format YYYYMMDD
parent_dir = os.path.dirname(r'Z:\HTOC\Data_Analytics\I_W\20231121')
date_folders = [f for f in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, f)) and f.isdigit() and len(f) == 8]
if date_folders:
    pptx_dir = os.path.join(parent_dir, sorted(date_folders)[-1])
else:
    pptx_dir = r'Z:\HTOC\Data_Analytics\I_W\20231121'
pptx_files = [f for f in os.listdir(pptx_dir) if f.lower().endswith('.pptx')]

keywords = ['CDC', 'NIH', 'FDA', 'HRSA', 'VA', 'CMS', 'IHS','DHA']
filtered_pptx_files = [f for f in pptx_files if any(k in f for k in keywords)]
print(filtered_pptx_files)


['CMS_Slide_20250506.pptx', 'FDA_Partner Input and I_W_Slide_20250506.pptx', 'IHS Biweekly - 05-06-2025.pptx', 'Partner Input and I_W_Slide_20250506 - DHA.pptx', 'Partner Input and I_W_Slide_20250506.CIR_final_VA.pptx', 'Partner Input and I_W_Slide_20250506_CDC.pptx', 'Partner Input and I_W_Slide_20250506_NIH.pptx']


In [6]:
from pptx import Presentation
import pandas as pd
import re
from datetime import datetime

def extract_text_and_tables_from_pptx(pptx_path):
    prs = Presentation(pptx_path)
    slides_content = []
    tables = []
    ip_like_data = []
    htoc_like_data = []
    htoc_pattern = re.compile(r'HTOC-\d{8}-\d{4}-[A-Z]')
    for slide in prs.slides:
        slide_text = []
        for shape in slide.shapes:
            if hasattr(shape, "text"):
                slide_text.append(shape.text)
                # Extract HTOC-like data from slide text
                htoc_matches = htoc_pattern.findall(shape.text)
                htoc_like_data.extend(htoc_matches)
            if shape.has_table:
                table = shape.table
                table_data = []
                for row in table.rows:
                    row_data = [cell.text for cell in row.cells]
                    table_data.append(row_data)
                    # Extract data matching the format: nnn.nnn.nnn[.]nnn
                    for cell_text in row_data:
                        if (
                            "[" in cell_text and
                            "." in cell_text and
                            "]" in cell_text and
                            any(char.isdigit() for char in cell_text)
                        ):
                            ip_like_data.append(cell_text.strip())
                        # Extract HTOC-like data from table cells
                        htoc_matches = htoc_pattern.findall(cell_text)
                        htoc_like_data.extend(htoc_matches)
                tables.append(table_data)
        slides_content.append('\n'.join(slide_text))
    # Only return unique values in the desired format
    ip_like_data = list({item for item in ip_like_data if "[" in item and "]" in item and "." in item})
    htoc_like_data = list(set(htoc_like_data))
    return slides_content, tables, ip_like_data, htoc_like_data

# Example usage:
# Extract from the 2nd slide of each pptx in filtered_pptx_files
slides_content = []
tables = []
ip_like_data = []
htoc_like_data = []

# Define htoc_pattern before usage
htoc_pattern = re.compile(r'HTOC-\d{8}-\d{4}-[A-Z]')

for pptx_file in filtered_pptx_files:
    # Determine the keyword for this pptx_file
    keyword_found = None
    for keyword in keywords:
        if keyword in pptx_file:
            keyword_found = keyword
            break

    prs = Presentation(os.path.join(pptx_dir, pptx_file))
    if len(prs.slides) >= 2:
        slide = prs.slides[1]  # 2nd slide (0-based index)
        slide_text = []
        slide_tables = []
        for shape in slide.shapes:
            if hasattr(shape, "text"):
                slide_text.append(shape.text)
                htoc_matches = htoc_pattern.findall(shape.text)
                htoc_like_data.extend(htoc_matches)
            if shape.has_table:
                table = shape.table
                table_data = []
                for row in table.rows:
                    row_data = [cell.text for cell in row.cells]
                    table_data.append(row_data)
                    for cell_text in row_data:
                        if (
                            "[" in cell_text and
                            "." in cell_text and
                            "]" in cell_text and
                            any(char.isdigit() for char in cell_text)
                        ):
                            # Store IP with keyword as tuple
                            ip_like_data.append((cell_text.strip(), keyword_found))
                        htoc_matches = htoc_pattern.findall(cell_text)
                        htoc_like_data.extend(htoc_matches)
                slide_tables.append(table_data)
        slides_content.append('\n'.join(slide_text))
        tables.extend(slide_tables)

# Only return unique values in the desired format
ip_like_data = list({item for item in ip_like_data if "[" in item and "]" in item and "." in item})
htoc_like_data = list(set(htoc_like_data))

# Assign each HTOC-like value with all associated IP-like values (grouping IPs under the most recent HTOC in the table)
htoc_ip_pairs = []
for table in tables:
    current_htoc = None
    for row in table:
        # Check for HTOC in the row
        htoc_matches = [cell for cell in row if htoc_pattern.match(cell)]
        if htoc_matches:
            current_htoc = htoc_matches[0]
        # Find IPs in the row
        ip_matches = [
            cell for cell in row
            if "[" in cell and "." in cell and "]" in cell and any(char.isdigit() for char in cell)
        ]
        # If there's a current HTOC, pair it with all IPs in this row and include the correct keyword for this table
        if current_htoc and ip_matches:
            # Find the index of the current table in tables
            table_idx = tables.index(table)
            # Use the corresponding pptx_file to get the keyword
            pptx_file = filtered_pptx_files[table_idx]
            keyword_found = None
            for keyword in keywords:
                if keyword in pptx_file:
                    keyword_found = keyword
                    break
            for ip in ip_matches:
                htoc_ip_pairs.append({
                    'HTOC_Like_Data': current_htoc,
                    'IP_Like_Data': ip,
                    'Keyword': keyword_found
                })

# Extract the date from each pptx_file's folder and add as a new column in the format MM/DD/YYYY

# Get the date string from pptx_dir (last 8 chars)
folder_date_str = os.path.basename(pptx_dir)
folder_date = datetime.strptime(folder_date_str, "%Y%m%d")
date_str = folder_date.strftime("%m/%d/%Y")

# Map disposition columns from the table header row (2nd row in each table)
disposition_headers = [
    'Unsuccessful Security Event.\nIndicator added to security controls.',
    'Successful Security Event.\nIndicator added to security controls.',
    'Indicator Not Malicious in Nature. Indicator observed, but not malicious.  E.g. IP address was not associated with malicious activity in this instance.',
    'Indicator Not Observed in Environment',
    'Indicator Blocked by Security Controls',
    'Not Listed/\x0bComment'
]

# Prepare a mapping from disposition header to column index for each table
disposition_col_indices = []
for table in tables:
    if len(table) > 1:
        header_row = table[1]
        indices = {}
        for disp in disposition_headers:
            try:
                indices[disp] = header_row.index(disp)
            except ValueError:
                indices[disp] = None
        disposition_col_indices.append(indices)
    else:
        disposition_col_indices.append({disp: None for disp in disposition_headers})
        
htoc_ip_df = pd.DataFrame(htoc_ip_pairs)
htoc_ip_df['Date'] = date_str

# Add disposition columns to the dataframe
disposition_values = []
for idx, row in htoc_ip_df.iterrows():
    htoc = row['HTOC_Like_Data']
    ip = row['IP_Like_Data']
    keyword = row['Keyword']
    # Find the table for this keyword (tables are in the same order as filtered_pptx_files)
    try:
        table_idx = filtered_pptx_files.index([f for f in filtered_pptx_files if keyword in f][0])
        table = tables[table_idx]
        indices = disposition_col_indices[table_idx]
        # Find the row in the table matching this HTOC and IP
        found_row = None
        for trow in table[2:]:  # skip header rows
            htoc_cell = trow[0] if len(trow) > 0 else ''
            ip_cell = trow[1] if len(trow) > 1 else ''
            # HTOC may be empty in some rows, so keep the last seen HTOC
            if htoc_cell:
                current_htoc = htoc_cell
            if current_htoc == htoc and ip_cell == ip:
                found_row = trow
                break
        # Extract disposition values for this row
        disp_dict = {}
        for disp, col_idx in indices.items():
            if found_row and col_idx is not None and col_idx < len(found_row):
                disp_dict[disp] = found_row[col_idx]
            else:
                disp_dict[disp] = ''
        disposition_values.append(disp_dict)
    except Exception:
        # If not found, fill with empty values
        disposition_values.append({disp: '' for disp in disposition_headers})

# Add each disposition as a column
for disp in disposition_headers:
    htoc_ip_df[disp] = [d[disp] for d in disposition_values]

# Replace 'X' or 'x' in disposition columns with their corresponding index (1-5), leave empty otherwise
for disp_idx, disp in enumerate(disposition_headers, start=1):
    htoc_ip_df[disp] = [
        str(disp_idx) if val.strip().lower() == 'x' else (val if val.strip() and val != 'X' and val != 'x' else '')
        for val in htoc_ip_df[disp]
    ]

display(htoc_ip_df)


,HTOC_Like_Data,IP_Like_Data,Keyword,Date,Unsuccessful Security Event.\nIndicator added to security controls.,Successful Security Event.\nIndicator added to security controls.,"Indicator Not Malicious in Nature. Indicator observed, but not malicious. E.g. IP address was not associated with malicious activity in this instance.",Indicator Not Observed in Environment,Indicator Blocked by Security Controls,Not Listed/ Comment
0,HTOC-20250414-0946-C,190.92.174[.]87,CMS,05/06/2025,,,,4,,
1,HTOC-20250422-0945-A,194.180.49[.]70,CMS,05/06/2025,,,,,5,
2,HTOC-20250422-0945-A,194.180.49[.]219,CMS,05/06/2025,,,,,5,
3,HTOC-20250422-0945-A,194.180.49[.]218,CMS,05/06/2025,,,,,5,
4,HTOC-20250422-0945-A,194.180.49[.]71,CMS,05/06/2025,,,,,5,
5,HTOC-20250428-0932-A,70.39.120[.]10,CMS,05/06/2025,,,,4,,
6,HTOC-20250414-0946-C,190.92.174[.]87,FDA,05/06/2025,,,,,5,
7,HTOC-20250422-0945-A,194.180.49[.]70,FDA,05/06/2025,,,,,5,
8,HTOC-20250422-0945-A,194.180.49[.]219,FDA,05/06/2025,,,,,5,
9,HTOC-20250422-0945-A,194.180.49[.]218,FDA,05/06/2025,,,,,5,


In [11]:
import pandas as pd
from openpyxl import load_workbook

file_path = r'Z:\HTOC\Data_Analytics\I_W\MasterProcessing\Master.xlsx'
sheet_name = 'Master Sheet'

# Define the columns for the master sheet
columns = [
    "Partner", "I_W Description", "I_W#", "Indicator Disposition Code",
    "Indicator Disposition Code Description", "Secondary Indicator Disposition Code",
    "Secondary Indicator Disposition Code Description", "Tertiary Indicator Disposition Code",
    "Tertiary Indicator Disposition Code Description", "Comment", "Bi-Weekly Date",
    "Technique", "Malware", "Threat Actor", "Aliases", "Vulnerability", "Actor Tag",
    "Sector", "Real Organization", "Threat Actor Country", "I&W Serial", "Affected Partners"
]

# Load or create the master dataframe
try:
    df = pd.read_excel(file_path, sheet_name=sheet_name)
except FileNotFoundError:
    df = pd.DataFrame(columns=columns)
if df.empty:
    df = pd.DataFrame(columns=columns)
else:
    missing_cols = [col for col in columns if col not in df.columns]
    for col in missing_cols:
        df[col] = ""
    df = df[columns]

# Process and append new rows from htoc_ip_pairs and htoc_ip_df
added = set()
for pair in htoc_ip_pairs:
    ip_htoc = pair['HTOC_Like_Data']
    ip = pair['IP_Like_Data']
    keyword_found = pair['Keyword']
    unique_key = (ip_htoc, ip, keyword_found)
    if unique_key in added:
        continue
    added.add(unique_key)
    match = htoc_ip_df[
        (htoc_ip_df['HTOC_Like_Data'] == ip_htoc) &
        (htoc_ip_df['IP_Like_Data'] == ip) &
        (htoc_ip_df['Keyword'] == keyword_found)
    ]
    if not match.empty:
        disp_number = ""
        disp_desc = ""
        for idx, disp_col in enumerate(disposition_headers, start=1):
            val = match.iloc[0][disp_col]
            if val.strip():
                disp_number = str(idx)
                disp_desc = disp_col.replace('\x0b', ' ').replace('\n', ' ').strip()
                break
    else:
        disp_number = ""
        disp_desc = ""
    test_input = [
        keyword_found,
        ip_htoc,
        ip,
        int(disp_number) if disp_number.isdigit() else None,
        disp_desc,
        "", "", "", "", "", date_str
    ]
    if len(test_input) < len(columns):
        test_input += [""] * (len(columns) - len(test_input))
    new_row = dict(zip(columns, test_input))
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

df.drop_duplicates(subset=["Partner", "I_W Description", "I_W#"], inplace=True, ignore_index=True)

# Write to Excel and auto-adjust column widths
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(file_path)
ws = wb[sheet_name]
for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter
    for cell in col:
        try:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        except:
            pass
    adjusted_width = (max_length + 2)
    ws.column_dimensions[col_letter].width = adjusted_width
wb.save(file_path)
print("Data saved to Excel with auto-spaced columns.")

Data saved to Excel with auto-spaced columns.
